# Issue #4 — Conferência das mesorregiões (Eric)

**Objetivo:** validar o alvo `Mesorregiao` derivado de `Cidade`, verificar cobertura, variações de nomes, 15 municípios amostrados, contagens por posto e alterações necessárias.

**Referência oficial:** [Condepe/Fidem — Mesorregiões de Pernambuco (classificação histórica do IBGE)](https://www.cf.pe.gov.br/observatorio/mapas/regioes/mesorregioes), consultada em 23/11/2026; consultar também [IBGE — Divisão Regional do Brasil em Mesorregiões e Microrregiões, páginas 52–55](https://www.ibge.gov.br/biblioteca/visualizacao/monografias/GEBIS%20-%20RJ/DRB/Divisao%20regional_v01.pdf).

> Importante: as **mesorregiões históricas** são o alvo do projeto. O IBGE adotou **regiões geográficas intermediárias e imediatas** em 2017; não misturar as duas classificações.

> **Correções aplicadas** ao `src/meso_map.py`: Afrânio, Dormentes, Itacuruba, Terra Nova → São Francisco Pernambucano; Feira Nova e Machados → Agreste Pernambucano. A fonte já confirma as seis correções. Este notebook lê a versão corrigida do mapeamento.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Funciona tanto executado na pasta raiz quanto na pasta notebooks/.
ROOT = Path.cwd() if (Path.cwd() / 'src' / 'data.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from data import carregar
from meso_map import meso_map

arquivo = next((ROOT / 'data' / 'raw').glob('*.csv'))
df = carregar(arquivo)
print(f'Registros: {len(df)} | Cidades (strings distintas): {df.Cidade.nunique()} | Códigos de posto: {df["Código"].nunique()}')
print('Linhas sem mesorregião:', df.Mesorregiao.isna().sum())
assert df.Mesorregiao.notna().all(), 'Há registros sem rótulo!'


Registros: 5104 | Cidades (strings distintas): 186 | Códigos de posto: 319
Linhas sem mesorregião: 0


## 1. Cobertura de cidades

Uma cidade com nome não encontrado no dicionário produziria um alvo nulo. Aqui também exibimos os nomes sem correspondência, caso a base mude.

In [2]:
nomes_dataset = set(df['Cidade'].unique())
sem_mapeamento = sorted(nomes_dataset - set(meso_map))
print('Nomes distintos do CSV:', len(nomes_dataset))
print('Nomes não mapeados:', sem_mapeamento)
assert not sem_mapeamento


Nomes distintos do CSV: 186
Nomes não mapeados: []


## 2. Variações de grafia

O dicionário conserva variantes de grafia que aparecem nas fontes. Nomes iguais após normalização precisam apontar para a mesma mesorregião.

In [3]:
pares_variantes = [
    ('Itaíba', 'Itaiba'), ('Calçado', 'Calçados'),
    ('Lagoa do Itaenga', 'Lagoa de Itaenga'),
    ('São Vicente Férrer', 'São Vicente Ferrer'),
    ('Camocim de São Felix', 'Camocim de São Félix'),
    ('Iguaraci', 'Iguaracy'), ('Jataúba', 'Jatauba'),
    ('Santa Maria do Cambucá', 'Santa Maria do Cambuca'),
]
variantes = pd.DataFrame([
    {'nome_1': a, 'nome_2': b, 'meso_1': meso_map.get(a), 'meso_2': meso_map.get(b),
     'linhas_nome_1': int((df.Cidade == a).sum()), 'linhas_nome_2': int((df.Cidade == b).sum()),
     'consistentes': meso_map.get(a) == meso_map.get(b)}
    for a, b in pares_variantes
])
display(variantes)
assert variantes.consistentes.all()


,nome_1,nome_2,meso_1,meso_2,linhas_nome_1,linhas_nome_2,consistentes
0,Itaíba,Itaiba,Agreste Pernambucano,Agreste Pernambucano,0,32,True
1,Calçado,Calçados,Agreste Pernambucano,Agreste Pernambucano,0,16,True
2,Lagoa do Itaenga,Lagoa de Itaenga,Mata Pernambucana,Mata Pernambucana,0,32,True
3,São Vicente Férrer,São Vicente Ferrer,Agreste Pernambucano,Agreste Pernambucano,16,0,True
4,Camocim de São Felix,Camocim de São Félix,Agreste Pernambucano,Agreste Pernambucano,16,0,True
5,Iguaraci,Iguaracy,Sertão Pernambucano,Sertão Pernambucano,16,0,True
6,Jataúba,Jatauba,Agreste Pernambucano,Agreste Pernambucano,16,16,True
7,Santa Maria do Cambucá,Santa Maria do Cambuca,Agreste Pernambucano,Agreste Pernambucano,0,16,True


## 3. Validação de 15 municípios

Amostra reprodutível de **15 nomes de cidades distintos**, selecionados com `random_state=42`. O campo `mesorregiao_referencia` foi transcrito da lista oficial Condepe/Fidem/IBGE, independentemente do dicionário do projeto; não é derivado de `meso_map`.

In [4]:
# Amostragem aleatória feita na base original (antes das correções)
amostra = sorted(df.Cidade.drop_duplicates().sample(15, random_state=42).tolist())
referencia_15 = {
    'Orobó': 'Agreste Pernambucano',
    'Surubim': 'Agreste Pernambucano',
    'Taquaritinga do Norte': 'Agreste Pernambucano',
    'Lagoa Grande': 'São Francisco Pernambucano',
    'Lagoa dos Gatos': 'Agreste Pernambucano',
    'Barreiros': 'Mata Pernambucana',
    'Tupanatinga': 'Agreste Pernambucano',
    'Calumbi': 'Sertão Pernambucano',
    'Paranatama': 'Agreste Pernambucano',
    'São João': 'Agreste Pernambucano',
    'Bonito': 'Agreste Pernambucano',
    'Ingazeira': 'Sertão Pernambucano',
    'São José do Egito': 'Sertão Pernambucano',
    'Panelas': 'Agreste Pernambucano',
    'Granito': 'Sertão Pernambucano',
}
assert set(amostra) == set(referencia_15), 'Amostra mudou; conferir fonte e referências novamente.'
validacao = pd.DataFrame({'Cidade':amostra})
validacao['mapeamento_projeto'] = validacao.Cidade.map(meso_map)
validacao['mesorregiao_referencia'] = validacao.Cidade.map(referencia_15)
validacao['correto'] = validacao.mapeamento_projeto.eq(validacao.mesorregiao_referencia)
display(validacao)
print('Acertos na amostra:', validacao.correto.sum(), 'de', len(validacao))
assert validacao.correto.all()


,Cidade,mapeamento_projeto,mesorregiao_referencia,correto
0,Barreiros,Mata Pernambucana,Mata Pernambucana,True
1,Bonito,Agreste Pernambucano,Agreste Pernambucano,True
2,Calumbi,Sertão Pernambucano,Sertão Pernambucano,True
3,Granito,Sertão Pernambucano,Sertão Pernambucano,True
4,Ingazeira,Sertão Pernambucano,Sertão Pernambucano,True
5,Lagoa Grande,São Francisco Pernambucano,São Francisco Pernambucano,True
6,Lagoa dos Gatos,Agreste Pernambucano,Agreste Pernambucano,True
7,Orobó,Agreste Pernambucano,Agreste Pernambucano,True
8,Panelas,Agreste Pernambucano,Agreste Pernambucano,True
9,Paranatama,Agreste Pernambucano,Agreste Pernambucano,True


Acertos na amostra: 15 de 15


## 4. Correções obrigatórias identificadas pela consulta à lista completa

O fato de a amostra ter 15 acertos **não significa** que o mapeamento completo estivesse correto. A comparação das listas das regiões detectou **6 municípios originalmente classificados em outra mesorregião**. A tabela abaixo documenta a correção, suas linhas e os rótulos atuais.

In [5]:
correcoes = {
    'Afrânio': ('Sertão Pernambucano', 'São Francisco Pernambucano'),
    'Dormentes': ('Sertão Pernambucano', 'São Francisco Pernambucano'),
    'Itacuruba': ('Sertão Pernambucano', 'São Francisco Pernambucano'),
    'Terra Nova': ('Sertão Pernambucano', 'São Francisco Pernambucano'),
    'Feira Nova': ('Mata Pernambucana', 'Agreste Pernambucano'),
    'Machados': ('Mata Pernambucana', 'Agreste Pernambucano'),
}
tabela_correcoes = pd.DataFrame([
    {'Cidade': cidade, 'antes': antes, 'depois': depois,
     'registros_afetados': int((df.Cidade == cidade).sum()),
     'postos_afetados': int(df.loc[df.Cidade == cidade, 'Código'].nunique()),
     'ajuste_aplicado': meso_map[cidade] == depois}
    for cidade, (antes, depois) in correcoes.items()
])
display(tabela_correcoes)
print('Registros corrigidos:', tabela_correcoes.registros_afetados.sum())
assert tabela_correcoes.ajuste_aplicado.all()


,Cidade,antes,depois,registros_afetados,postos_afetados,ajuste_aplicado
0,Afrânio,Sertão Pernambucano,São Francisco Pernambucano,32,2,True
1,Dormentes,Sertão Pernambucano,São Francisco Pernambucano,32,2,True
2,Itacuruba,Sertão Pernambucano,São Francisco Pernambucano,16,1,True
3,Terra Nova,Sertão Pernambucano,São Francisco Pernambucano,16,1,True
4,Feira Nova,Mata Pernambucana,Agreste Pernambucano,16,1,True
5,Machados,Mata Pernambucana,Agreste Pernambucano,16,1,True


Registros corrigidos: 128


## 5. Distribuição dos registros e postos

As contagens esperadas na issue refletem o dicionário **antes** da conferência. Depois da correção, elas devem mudar; o total de linhas permanece 5.104.

In [6]:
resumo = (df.groupby('Mesorregiao')
          .agg(registros=('Código', 'size'), postos=('Código', 'nunique'),
               cidades_strings=('Cidade', 'nunique'))
          .sort_values('registros', ascending=False))
display(resumo)
assert resumo.registros.sum() == 5104
assert resumo.postos.sum() == df['Código'].nunique()

postos_por_cidade = (df.groupby(['Mesorregiao','Cidade'])['Código']
                    .nunique().rename('quantidade_postos').reset_index()
                    .sort_values('quantidade_postos', ascending=False))
print('Municípios/nomes com mais de um posto:', int((postos_por_cidade.quantidade_postos > 1).sum()))
display(postos_por_cidade.query('quantidade_postos > 1').head(25))


,registros,postos,cidades_strings
Mesorregiao,,,
Agreste Pernambucano,1568,98,72
Sertão Pernambucano,1520,95,41
Mata Pernambucana,1024,64,43
Metropolitana do Recife,496,31,15
São Francisco Pernambucano,496,31,15


Municípios/nomes com mais de um posto: 64


,Mesorregiao,Cidade,quantidade_postos
131,Sertão Pernambucano,Araripina,14
162,Sertão Pernambucano,Sertânia,8
128,Metropolitana do Recife,Recife,6
64,Agreste Pernambucano,Taquaritinga do Norte,6
172,São Francisco Pernambucano,Belém de São Francisco,5
160,Sertão Pernambucano,Serra Talhada,5
19,Agreste Pernambucano,Caruaru,5
157,Sertão Pernambucano,Santa Cruz da Venerada,4
117,Metropolitana do Recife,Cabo,4
165,Sertão Pernambucano,São José do Egito,4


## 6. Conclusão e limitações

- **Cobertura:** 0/5.104 linhas sem rótulo.
- **Variantes:** os pares de grafia examinados são consistentes.
- **Amostra:** 15/15 nomes conferidos concordam com a classificação histórica oficial.
- **Correções:** 6 entradas no dicionário foram corrigidas, com impacto nas contagens de classe.
- **Postos:** o CSV contém 319 valores de `Código` distintos, em contraste com os 310 mencionados na documentação anterior; o número deve ser revisado pelo grupo.
- **Critério territorial:** mesorregiões históricas do IBGE, não as divisões geográficas vigentes a partir de 2017.
- **Limite:** a validação da amostra e a comparação das listas identificam problemas de atribuição; **não** auditam se o nome do posto da APAC está correto na origem. O dicionário ainda contém grafias alternativas e nomes históricos que devem permanecer documentados.

**Aviso ao grupo:** como o alvo `y` mudou em algumas linhas, recomenda-se regenerar os resultados dos 360 experimentos e as métricas do modelo com o dicionário corrigido. Não divulgar resultados antigos como se fossem da base revisada.